# (Alternative) Wan 2.1 1.3B Text-to-Video Model
# UNVERIFIED on local hardware without GPU. Requires ~12-16GB VRAM.
# Uncomment to use Wan 2.1 with Diffusers WanPipeline:
# from diffusers import AutoencoderKLWan, WanPipeline
# model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
# vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
# pipeline = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
# pipeline.enable_model_cpu_offload()

In [ ]:
# Step 1: Verify Colab GPU is available (T4 with 15GB VRAM)
!nvidia-smi

In [ ]:
# Step 2: Install pinned open-source dependencies
!pip install --quiet \
    diffusers==0.32.1 \
    transformers==4.47.1 \
    accelerate==1.2.1 \
    sentencepiece==0.2.0 \
    imageio[ffmpeg]==2.36.1 \
    safetensors==0.4.5

In [ ]:
# Step 3: Upload scene_plan.json and clip_helpers.py
import os
from pathlib import Path

if not Path("scene_plan.json").exists() or not Path("clip_helpers.py").exists():
    try:
        from google.colab import files
        print("Please upload your scene_plan.json and clip_helpers.py files:")
        uploaded = files.upload()
    except ImportError:
        print("Running outside Google Colab: ensure scene_plan.json and clip_helpers.py are in the working directory.")

In [ ]:
# Step 4: Load and validate scene plan
from clip_helpers import (
    load_scene_plan,
    filter_pending_scenes,
    derive_scene_seed,
    get_scene_filename,
    create_clips_zip,
    validate_clip_file,
)

plan_data = load_scene_plan("scene_plan.json")
scenes = plan_data.get("scenes", [])
print(f"Plan ID: {plan_data.get('plan_id', 'unknown')}")
print(f"Total scenes: {len(scenes)}")
print(f"Total planned video duration: {plan_data.get('total_duration_seconds', 0.0):.1f}s")

output_dir = Path("generated_clips")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Step 5: Load LTX-Video Pipeline (Default T4 low-VRAM open model)
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

print("Loading LTX-Video model in bfloat16 with CPU offload...")
pipeline = LTXPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16,
)
pipeline.enable_model_cpu_offload()
print("Model successfully loaded and ready for inference.")

### Alternative Model: Wan 2.1 (1.3B)
If you prefer to use the Wan 2.1 1.3B open model instead of LTX-Video, uncomment and run the cell below instead of Step 5.
> **Note:** UNVERIFIED on local hardware without GPU. Diffusers repository ID: `Wan-AI/Wan2.1-T2V-1.3B-Diffusers`.

In [ ]:
# (Alternative) Wan 2.1 1.3B Text-to-Video Model
# UNVERIFIED on local hardware without GPU. Requires ~12-16GB VRAM.
# Uncomment to use Wan 2.1 with Diffusers WanPipeline:
# from diffusers import AutoencoderKLWan, WanPipeline
# model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
# vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
# pipeline = WanPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
# pipeline.enable_model_cpu_offload()

In [ ]:
# Step 6: Test One Clip First (scene_001)
# UNVERIFIED on local CPU environment. Requires Colab T4 GPU.
first_scene = scenes[0]
first_filename = get_scene_filename(first_scene)
first_output_path = output_dir / first_filename

print(f"Generating test clip: {first_filename}")
print(f"Visual Prompt: {first_scene['visual_prompt']}")

test_seed = derive_scene_seed(base_seed=42, scene_index=0)
generator = torch.Generator(device="cuda").manual_seed(test_seed)

# Target frames: LTX-Video requires num_frames to satisfy the 8n + 1 formula
dur_sec = float(first_scene.get("target_clip_seconds", 5.0))
raw_frames = int(round(dur_sec * 24))
n_steps = max(3, min(15, round((raw_frames - 1) / 8)))
num_frames = 8 * n_steps + 1

video_frames = pipeline(
    prompt=first_scene["visual_prompt"],
    negative_prompt=first_scene.get("negative_prompt", ""),
    width=704,
    height=480,
    num_frames=num_frames,
    num_inference_steps=30,
    generator=generator,
).frames[0]

export_to_video(video_frames, str(first_output_path), fps=24)
print(f"Test clip created: {first_output_path} (size: {first_output_path.stat().st_size} bytes)")

In [ ]:
# Step 7: Batch Generation with Resume Support
# UNVERIFIED on local CPU environment. Requires Colab T4 GPU.
pending_scenes = filter_pending_scenes(scenes, output_dir, overwrite=False)
print(f"Remaining scenes to generate: {len(pending_scenes)} of {len(scenes)}")

for sc in pending_scenes:
    sc_id = sc.get("id", f"scene_{sc.get('scene_index', 0)+1:03d}")
    fname = get_scene_filename(sc)
    out_path = output_dir / fname
    
    idx = int(sc.get("scene_index", 0))
    sc_seed = derive_scene_seed(base_seed=42, scene_index=idx)
    generator = torch.Generator(device="cuda").manual_seed(sc_seed)
    
    dur_sec = float(sc.get("target_clip_seconds", 5.0))
    raw_frames = int(round(dur_sec * 24))
    n_steps = max(3, min(15, round((raw_frames - 1) / 8)))
    num_frames = 8 * n_steps + 1
    
    print(f"--> Rendering {sc_id} ({dur_sec:.1f}s, {num_frames} frames)...")
    frames = pipeline(
        prompt=sc["visual_prompt"],
        negative_prompt=sc.get("negative_prompt", ""),
        width=704,
        height=480,
        num_frames=num_frames,
        num_inference_steps=30,
        generator=generator,
).frames[0]
    
    export_to_video(frames, str(out_path), fps=24)
    print(f"    Saved {fname} ({out_path.stat().st_size} bytes)")

print("All planned scene clips rendered successfully!")

In [ ]:
# Step 8: Package into ZIP archive and download
zip_destination = Path("ai_clips.zip")
created_zip = create_clips_zip(output_dir, zip_destination)
print(f"Archive ready: {created_zip} (size: {created_zip.stat().st_size} bytes)")

try:
    from google.colab import files
    files.download(str(created_zip))
    print("Download initiated to your local computer.")
except ImportError:
    print(f"Clips archived at {created_zip.resolve()}")